## S-LDSC step1

## making the list of file names

In [ ]:
# ---- Set the following paths to match your environment ----
data_path  = "/path/to/data"            # directory for data
# -----------------------------------------------------------

data_f=${data_path}/CC_10percent_SNP_noexp_2
cd ${data_f}
mkdir ${data_f}/info
(ls *.bed.gz | sed -e 's/.bed.gz//g')> ${data_f}/info/data_list

data_f=${data_path}/CC_10percent_SNP_noexp_2_EAS
cd ${data_f}
mkdir ${data_f}/info
(ls *.bed.gz | sed -e 's/.bed.gz//g')> ${data_f}/info/data_list

## calculate LDscore and S-LDSC

In [ ]:
# ---- Set the following paths to match your environment ----
wd_path  = "/path/to/shell_script"      # directory of shell script
LDSCORE_path = "/path/to/reference/LDSCORE"     # directory of LDSC reference files (1000 Genomes Phase 3, EAS population)
GWAS_path = "/path/to/reference/GWAS"     # GWAS sumstats
# -----------------------------------------------------------

export MKL_NUM_THREADS=1
export OMP_NUM_THREADS=1
export MKL_DOMAIN_NUM_THREADS=1
unset PROMPT_COMMAND



cd ${wd_path}
mkdir -p log

dd=${data_path}/CC_10percent_SNP_noexp_2

info=${dd}/info/data_list
num=`wc -l $info |  cut -d " " -f1`


#####STEP1-1: LD Score (EUR)
 #MAIN JOB

qsub -pe def_slot 1 \
   -l s_vmem=10G,mem_req=10G \
   -cwd \
   -t 1:${num} -tc 50 \
   -o log/sldsc_eur.log \
   -e log/sldsc_eur.error \
   ${wd_path}/ldscore_eur_ver1.sh ${dd} ${LDSCORE_path}


dd=${data_path}/CC_10percent_SNP_noexp_2

info=${dd}/info/data_list
num=`wc -l $info |  cut -d " " -f1`


#####STEP1-2: S-LDSC (EUR)
 #MAIN JOB

qsub -pe def_slot 1 \
   -l s_vmem=10G,mem_req=10G \
   -cwd \
   -t 1:${num} -tc 100 \
   -o log/sldsc_eur_2.log \
   -e log/sldsc_eur_2.error \
   ${wd_path}/sldsc_eur-alltrait.sh ${dd} ${LDSCORE_path} ${GWAS_path}





#####STEP2-1: LD Score (EAS)
 #MAIN JOB

qsub -pe def_slot 1 \
   -l s_vmem=10G,mem_req=10G \
   -cwd \
   -t 1:${num} -tc 50 \
   -o log/sldsc_eur.log \
   -e log/sldsc_eur.error \
   ${wd_path}/ldscore_eas.sh ${dd} ${LDSCORE_path}



#####STEP2-2: S-LDSC (EAS)
 #MAIN JOB

qsub -pe def_slot 1 \
   -l s_vmem=10G,mem_req=10G \
   -cwd \
   -t 1:${num} -tc 100 \
   -o log/sldsc_eur_2.log \
   -e log/sldsc_eur_2.error \
   ${wd_path}/sldsc_eas.sh ${dd} ${LDSCORE_path} ${GWAS_path}

